# Make the splits

A split decides **which charts to keep** and **which part (train/validation/test) each one goes to**. It scans every `.tja` under `raw/`, drops charts the parser can't handle or the filters exclude, and splits the rest by group, so every chart of a group lands in the same part.

This notebook makes the two splits used from here on:

| Split | Courses | Grouping | Parts |
|---|---|---|---|
| `oni_v2` | Oni | song folder (keeps covers together) | stratified by star level, 80/10/10, seed 42 |
| `all_v1` | all five | song folder | copied from `oni_v2`; songs without an Oni chart go to train |

So `all_v1` (for pretraining on all courses) never puts an `oni_v2` validation or test song in train, in any course.

Output goes to `splits/<name>/`: `config.json`, `manifest.csv` and `report.json`. Splits are never edited in place. If you change a filter, give the split a new name. `oni_v1` is left as it was.

In [ ]:
from donderformer.paths import DataPaths
from donderformer.data.data_splitter import DataSplitter

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for the toy dataset.
paths = DataPaths()

oni_splitter = DataSplitter(
    name="oni_v2",
    courses=("oni",),          # None = all; options: easy, normal, hard, oni, ura
    genres=None,               # None = all; top-level ESE folders, e.g. ("01 Pop", "02 Anime")
    bpm_range=None,            # e.g. (100, 220)
    level_range=None,          # e.g. (7, 10)
    ratios=(0.8, 0.1, 0.1),
    seed=42,
    group_by="song",           # "song" = <genre>/<song>/ folder; "audio" = one audio file (oni_v1)
    stratify_by_level=True,    # strata: <=6, 7, 8, 9, 10 stars (by the song's Oni level)
)

all_splitter = DataSplitter(
    name="all_v1",
    courses=None,
    ratios=(0.8, 0.1, 0.1),
    seed=42,
    group_by="song",
    song_parts_from="oni_v2",  # same validation/test songs as oni_v2
)

In [ ]:
oni_report = oni_splitter.run(paths)

In [ ]:
all_report = all_splitter.run(paths)

## Survival funnel

Charts left after each filter, in order. A chart dropped for several reasons is counted at the first one. Charts that later fail labelling (e.g. notes past the end of the audio) are listed by `02_extract_features` in `features/<extractor>/failures/<split>.json`.

In [ ]:
import json

def show_funnel(report, course=None):
    steps = report["funnel_by_course"][course] if course else report["funnel"]
    print(f"{report['name']}{' / ' + course if course else ''}")
    for step in steps:
        print(f"  {step['stage']:<26} -{step['dropped']:>6}  {step['remaining']:>6}")

show_funnel(all_report)                    # every course, all filters
show_funnel(all_report, "oni")
show_funnel(oni_report)                    # oni_v2: course filter applied

## Leak checks

`run()` already asserts that no song folder or audio file is in two parts, and for `all_v1` that every song kept the part it has in `oni_v2`. Titles found in more than one part are listed for a manual look: they're usually different songs with the same name.

In [ ]:
for report in (oni_report, all_report):
    print(report["name"], {k: v for k, v in report["leak_check"].items() if k != "titles_crossing"})
    for title, parts in report["leak_check"]["titles_crossing"].items():
        print("   same title in", parts, ":", title)
print(all_report["song_parts_from"])

## Report

Re-run from here to plot any existing split's `report.json`.

In [ ]:
import matplotlib.pyplot as plt

def plot_report(name):
    report = json.loads((paths.split_dir(name) / "report.json").read_text())
    panels = [("Kept per split", report["kept_by"]["split"]),
              ("Kept per course", report["kept_by"]["course"]),
              ("Kept per genre", report["kept_by"]["genre"])]
    if "lost_reasons" in report:
        panels.append(("Lost reasons (top 10)", dict(list(report["lost_reasons"].items())[:10])))

    fig, axes = plt.subplots(1, len(panels) + 2, figsize=(5 * (len(panels) + 2), 4))
    for ax, (title, counts) in zip(axes, panels):
        ax.bar(list(counts), list(counts.values()))
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=60)

    # Level mix per part, as shares, so stratification is visible.
    ax = axes[-2]
    by_part = report["kept_by"].get("split_level", {})
    levels = sorted({lv for counts in by_part.values() for lv in counts}, key=lambda lv: (lv == "", int(lv) if lv else 0))
    for part, counts in by_part.items():
        total = sum(counts.values()) or 1
        ax.plot(levels, [counts.get(lv, 0) / total for lv in levels], marker="o", label=part)
    ax.set_title("Level share per part")
    ax.legend()

    axes[-1].hist(report["kept_bpm"], bins=30)
    axes[-1].set_title("BPM of kept charts")
    fig.suptitle(name)
    fig.tight_layout()
    plt.show()

plot_report("oni_v2")
plot_report("all_v1")